In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer : une Qiskit Function par Global Data Quantum


> **Note:** Les Qiskit Functions sont une fonctionnalité expérimentale réservée aux utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et peuvent évoluer.
# Vue d'ensemble
Le Quantum Portfolio Optimizer est une Qiskit Function qui s'attaque au problème d'optimisation dynamique de portefeuille, un problème classique en finance visant à rééquilibrer périodiquement des investissements sur un ensemble d'actifs afin de maximiser les rendements et de minimiser les risques. En déployant des techniques d'optimisation quantique de pointe, cette fonction simplifie le processus pour que les utilisateurs, sans expertise particulière en informatique quantique, puissent profiter de ses avantages dans la recherche de trajectoires d'investissement optimales. Idéal pour les gestionnaires de portefeuille, les chercheurs en finance quantitative et les investisseurs individuels, cet outil permet le backtesting de stratégies de trading dans le cadre de l'optimisation de portefeuille.
# Description de la fonction
Le Quantum Portfolio Optimizer utilise l'algorithme Variational Quantum Eigensolver (VQE) pour résoudre un problème d'optimisation binaire quadratique sans contrainte (QUBO), en traitant des problèmes d'optimisation dynamique de portefeuille. Il suffit de fournir les données de prix des actifs et de définir la contrainte d'investissement ; la fonction exécute ensuite le processus d'optimisation quantique et retourne un ensemble de trajectoires d'investissement optimisées.

Le processus comporte quatre étapes principales. D'abord, les données d'entrée sont transformées en un problème compatible quantique : on construit le QUBO du problème d'optimisation dynamique de portefeuille et on le convertit en un opérateur quantique (Hamiltonien d'Ising). Ensuite, le problème d'entrée et l'algorithme VQE sont adaptés pour s'exécuter sur le matériel quantique. Le VQE est alors exécuté sur ce matériel, et enfin, les résultats sont post-traités pour fournir les trajectoires d'investissement optimales. Le système intègre également un post-traitement tenant compte du bruit (basé sur [SQD](/guides/qiskit-addons-sqd)) afin de maximiser la qualité des résultats.

Cette Qiskit Function est basée sur le [manuscrit publié](https://arxiv.org/abs/2412.19150) par Global Data Quantum.
![Visualisation du flux de travail de la fonction](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Entrée
Les arguments d'entrée de la fonction sont décrits dans le tableau suivant. Les données des actifs et les autres spécifications du problème sont obligatoires ; les paramètres VQE peuvent être ajoutés pour personnaliser le processus d'optimisation.
| **Nom**               | **Type**           | **Description**                                          | **Requis** | **Défaut**                          | **Exemple**                              |
|------------------------|--------------------|----------------------------------------------------------|--------------|--------------------------------------|------------------------------------------|
| assets                 | json               | Dictionnaire avec les prix des actifs                         | Oui          | -                                    | -                                        |
| qubo_settings          | json               | Paramètres du QUBO                                     | Oui          | -                                    | Voir les exemples dans le tableau ci-dessous     |
| ansatz_settings        | json               | Paramètres de l'ansatz                                   | Non           | `None`                                | Voir les exemples dans le tableau ci-dessous.     |
| optimizer_settings     | json               | Paramètres de l'optimiseur                                | Non           | `None`                                | Voir les exemples dans le tableau ci-dessous.     |
| backend                | str                | Nom du backend QPU                                     | Non           |  -    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | Liste des identifiants de session pour récupérer des données des exécutions précédentes [(*)](#1-note) | Non           | Liste vide                           | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | Applique le post-traitement SQD tenant compte du bruit                      | Non           | `True`                                | `True`                                   |
| tags                   | list of strings    | Liste de tags pour identifier l'expérience                  | Non           | Liste vide                           | `["optimization", "quantum_computing"]`  |
<span id="1-note"></span>
*Pour reprendre une exécution ou récupérer des jobs traités lors d'une ou plusieurs sessions précédentes, la liste des identifiants de session doit être passée dans le paramètre `previous_session_id`. Cela est particulièrement utile lorsqu'une tâche d'optimisation n'a pas pu se terminer en raison d'une erreur, et que l'exécution doit être finalisée. Pour ce faire, tu dois fournir les mêmes arguments que lors de l'exécution initiale, ainsi que la liste `previous_session_id` telle que décrite.*
> **Caution:** Le chargement des données des sessions précédentes (pour reprendre une optimisation) peut prendre jusqu'à une heure.
## `assets`
Les données doivent être structurées sous forme d'objet JSON contenant les prix de clôture d'actifs financiers à des dates spécifiques. Le format est le suivant :

- Clé primaire (chaîne de caractères) : le nom ou le symbole boursier de l'actif financier (par exemple, "8801.T").
- Clé secondaire (chaîne de caractères) : la date au format AAAA-MM-JJ.
- Valeur (nombre) : le prix de clôture de l'actif à la date indiquée. Les prix peuvent être saisis normalisés ou non normalisés.

*Note : tous les dictionnaires doivent avoir la même clé secondaire (dates). Si un actif donné n'a pas de valeur pour une date présente chez les autres, les données doivent être complétées pour garantir la cohérence. Par exemple, on peut utiliser le dernier prix de clôture connu de cet actif.*
### Exemple
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** Les données des actifs doivent contenir au minimum les prix de clôture à ``(nt+1) * dt`` (voir la section d'entrée [`qubo_settings`](#qubo_settings)) horodatages (par exemple, des jours).
## `qubo_settings`
Le tableau suivant décrit les clés du dictionnaire `qubo_settings`. Construis le dictionnaire en spécifiant le nombre de pas de temps `nt`, le nombre de qubits de résolution `nq` et la valeur `max_investment` — ou modifie d'autres valeurs par défaut.
| Nom                | Type    | Description                                                                 | Requis | Défaut | Exemple |
|---------------------|---------|-----------------------------------------------------------------------------|----------|---------|---------|
| nt                  | int     | Nombre de pas de temps                                                         | Oui      | -       | 4       |
| nq                  | int     | Nombre de qubits de résolution                                                                | Oui      | -       | 4       |
| max_investment      | float   | Nombre maximum d'unités de devise investies au total sur tous les actifs                           | Oui      | -       | 10      |
| dt*                  | int     | Fenêtre temporelle prise en compte à chaque pas de temps. L'unité correspond aux intervalles de temps entre les clés des données d'actifs                                                 | Non       | 30      | -       |
| risk_aversion       | float     | Coefficient d'aversion au risque                                   | Non       | 1000    | -       |
| transaction_fee     | float     | Coefficient de frais de transaction                                                 | Non       | 0.01   | -       |
| restriction_coeff   | float   | Multiplicateur de Lagrange utilisé pour imposer la contrainte du problème dans la formulation QUBO | Non       | 1       | -       |
## `ansatz_settings`
Pour modifier les options par défaut, crée un dictionnaire pour le paramètre `ansatz_settings` avec les clés suivantes. Par défaut, l'ansatz est réglé sur `"real_amplitudes"` et les deux options supplémentaires (voir le tableau ci-dessous) sont à `False`.
| Nom                  | Type     | Description                                                                 | Requis | Défaut |
|-----------------------|----------|-----------------------------------------------------------------------------|----------|---------|
| ansatz[*](#single-star)                | str      | Ansatz à utiliser                                             | Non      | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     | Active la sous-routine multiple passmanager (non disponible pour l'ansatz Tailored) | Non       | `False`   |
| dd_enable   | bool     | Ajoute le découplage dynamique                                    | Non       | `False`   |

<span id="single-star"></span>
\* Ansatzes disponibles
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (Uniquement pour le backend `ibm_torino`, 7 actifs, 4 pas de temps et 4 qubits de résolution)

<span id="double-star"></span>
** Si ``multiple_passmanager`` est réglé sur ``False``, la fonction utilise le pass manager Qiskit par défaut avec `optimization_level=3`. S'il est réglé sur ``True``, la sous-routine ``multiple_passmanager`` compare trois pass managers : le pass manager Qiskit par défaut précédent, un pass manager mappant les qubits sur la chaîne de premiers voisins du QPU, et les services du [transpileur IA](/guides/ai-transpiler-passes). Le pass manager dont l'erreur cumulée estimée est la plus faible est ensuite sélectionné.
## `optimizer_settings`
Ce paramètre est un dictionnaire avec certaines options configurables du processus d'optimisation.
| Nom               | Type   | Description                                     | Requis | Défaut               |
|--------------------|--------|-------------------------------------------------|----------|-----------------------|
| primitive_options  | json   | Paramètres de la primitive               | Non      | -                     |
| optimizer          | str    | Optimiseur classique sélectionné                            | Non       | `"differential_evolution"` |
| optimizer_options  | json   | Configuration de l'optimiseur                  | Non       | -                     |
> **Note:** Actuellement, le seul optimiseur disponible est `"differential_evolution"`.

Sous les clés `primitive_options` et `optimizer_options`, on définit des dictionnaires avec les paramètres suivants :
### `primitive_options`
| Nom              | Type   | Description                                | Requis | Défaut                    | Exemple                    |
|-------------------|--------|--------------------------------------------|----------|----------------------------|----------------------------|
| sampler_shots     | int    | Nombre de shots du Sampler.            | Non       | 100000                     | -                          |
| estimator_shots   | int    | Nombre de shots de l'Estimator.          | Non       | 25000                      | -                          |
| estimator_precision | float | Précision souhaitée de la valeur attendue. Si spécifiée, la précision sera utilisée à la place de `estimator_shots`. | Non | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int or str | Durée maximale pendant laquelle une session Runtime peut rester ouverte avant d'être fermée de force. Peut être donné en secondes (int) ou sous forme de chaîne, comme `"2h 30m 40s"`. Doit être inférieur au maximum imposé par le système. | Non | `None` | `"1h 15m"`                |
### `optimizer_options`
| Nom              | Type     | Description                              | Requis | Défaut       |
|-------------------|----------|------------------------------------------|----------|---------------|
| num_generations   | int      | Nombre de générations                    | Non       | 20            |
| population_size   | int      | Taille de la population                    | Non       | 20            |
| mutation_range    | list   | Facteur de mutation maximum et minimum              | Non       | [0, 0.25]     |
| recombination     | float      | Facteur de recombinaison                     | Non       | 0.4           |
| max_parallel_jobs | int      | Nombre maximum de jobs QPU exécutés en parallèle  | Non       | 3             |
| max_batchsize | int      | Taille maximale des lots  | Non       | 200     |
> **Note:** - Le nombre de générations évaluées par l'évolution différentielle est `num_generations` + 1, car la population initiale est incluse.
> - Le nombre total de circuits est calculé comme `(num_generations + 1) * population_size`.
> - Utiliser une population plus grande et davantage de générations améliore généralement la qualité des résultats d'optimisation. Cependant, il est déconseillé de dépasser une taille de population de 120 et un nombre de générations supérieur à 20 (par exemple, ``120 * 21 = 2520`` circuits au total), car cela générerait un nombre excessif de circuits, coûteux en calcul et en temps.
> 
> - La fonction permet de reprendre une optimisation précédente, et il est toujours possible d'augmenter le nombre de générations (en fournissant les mêmes entrées, sauf pour `previous_session_id` et un ``num_generations`` augmenté).
> **Note:** Assure-toi de respecter les limites de jobs Qiskit Runtime.
> 
> - Sampler : `sampler_shots <= 10_000_000`.
> - Estimator : `max_batchsize * estimator_shots * observable_size <= 10_000_000` (pour cette fonction, tous les termes de l'observable commutent, donc `observable_size=1`).
> 
> Consulte le guide [Limites de jobs](/guides/job-limits) pour plus d'informations.
# Sortie
La fonction retourne deux dictionnaires : le dictionnaire `"result"`, qui contient les meilleurs résultats d'optimisation, y compris la solution optimale et son coût objectif minimal associé ; et `"metadata"`, avec les données de tous les résultats obtenus au cours du processus d'optimisation, ainsi que leurs métriques respectives.

Le premier dictionnaire se concentre sur la solution la plus performante, tandis que le second fournit des informations détaillées sur toutes les solutions, notamment les coûts objectifs et d'autres métriques pertinentes.

## Dictionnaires de sortie :
| **Nom**    | **Type**                     | **Description**                                                                 | **Exemple**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | Contient la stratégie d'investissement dans le temps, chaque horodatage étant associé aux poids d'investissement par actif (chaque poids est le montant investi normalisé par le montant total investi). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | Données générées pendant l'analyse, incluant les solutions, les coûts et les métriques.    | Voir les exemples ci-dessous |
### Description du dictionnaire `metadata`
| **Nom**                 | **Type**              | **Description**                                                                                     | **Exemple**                  |
|--------------------------|-----------------------|-----------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | Identifiant unique de la session IBM Quantum.                          | `"d0h30qjvpqf00084fgw0"`        |
| all_samples_metrics | dict                 | Dictionnaire contenant diverses métriques pour chaque échantillon post-traité, telles que les coûts ou les contraintes. | Voir la description ci-dessous        |
| sampler_counts           | dict[str, int]        | Dictionnaire dont les clés sont des représentations binaires des solutions échantillonnées et les valeurs sont leurs comptages. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | Liste indiquant l'ordre d'investissement des actifs à chaque pas de temps dans les stratégies d'investissement. | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | Matrice QUBO du problème.                                                                         | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | Résumé des temps d'utilisation CPU et QPU (en secondes) à différentes étapes du processus.                | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### Description du dictionnaire `all_samples_metrics`
| **Nom**                | **Type**       | **Description**                                                                                                  | **Exemple**                  |
|-------------------------|----------------|------------------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | Stratégies d'investissement dérivées des états quantiques décodés. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | Nombre de fois que chaque trajectoire d'investissement a été échantillonnée. L'index correspond à `investment_trajectories`.                | `[5, 3]`                     |
| objective_costs         | list[float]    | Valeur de la fonction objectif pour chaque trajectoire d'investissement, triée du plus bas au plus élevé.                 | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | Performance ajustée au risque (ratio de Sharpe) pour chaque trajectoire d'investissement. Alignée par index.                      | `[1.1, 0.7]`                 |
| returns                 | list[float]    | Rendement attendu pour chaque trajectoire d'investissement. Aligné par index.                                               | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | Déviation maximale de la contrainte au sein de chaque trajectoire d'investissement. Alignée par index.                               | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | Coût de transaction estimé associé à chaque trajectoire d'investissement. Aligné par index.                        | `[0.01, 0.02]`               |
# Prise en main
Authentifie-toi avec ta [clé API](https://quantum.cloud.ibm.com/) et sélectionne la Qiskit Function comme suit. (Cet extrait suppose que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## Exemple : optimisation dynamique de portefeuille avec sept actifs
Cet exemple montre comment exécuter la fonction d'optimisation dynamique de portefeuille (DPO) et ajuster ses paramètres pour des performances optimales. Il inclut des étapes détaillées pour affiner les paramètres et atteindre les résultats souhaités.

Ce cas implique sept actifs, quatre pas de temps et quatre qubits de résolution, pour un besoin total de 112 qubits.
### 1. Lire les actifs inclus dans le portefeuille.
Si tous les actifs du portefeuille sont stockés dans un dossier à un chemin spécifique, tu peux les charger dans un `pandas.DataFrame` et le convertir en objet `dict` à l'aide de la fonction suivante.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

Pour cet exemple, nous avons utilisé les actifs [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) et [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). La figure suivante illustre les données utilisées dans cet exemple, montrant l'évolution du prix de clôture journalier des actifs du 1er janvier au 1er septembre 2023.

Dans cet exemple, pour garantir l'uniformité entre les dates, nous avons comblé les jours non ouvrés avec le prix de clôture de la dernière date disponible. Cette étape est nécessaire car les actifs sélectionnés proviennent de différents marchés avec des jours de bourse variables, et il est essentiel de standardiser le jeu de données pour assurer la cohérence.
![Visualisation des données historiques des actifs](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. Définir le problème.

Définis les spécifications du problème en configurant les paramètres du dictionnaire `qubo_settings`.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. Définir les paramètres de l'optimiseur et de l'ansatz. (Optionnel)
Définis optionnellement des exigences spécifiques pour le processus d'optimisation, notamment le choix de l'optimiseur et de ses paramètres, ainsi que la spécification de la primitive et de ses configurations.

Pour l'ansatz Tailored, la taille de population choisie est basée sur des expériences précédentes montrant que cette valeur produit une optimisation stable et efficace.

Dans le cas de l'ansatz Real Amplitudes, tu peux suivre une relation linéaire entre la ``population_size`` et le nombre de qubits dans le circuit. À titre de règle empirique approximative, il est recommandé d'utiliser une ``population_size`` **minimale** ``~ 0.8 * n_qubits`` pour l'ansatz `real_amplitudes`.

On s'attend à ce que l'Optimized Real Amplitudes offre de meilleures performances d'optimisation que l'ansatz Real Amplitudes. Cependant, le nombre de variables à optimiser dans cet ansatz augmente bien plus vite que dans le cas Real Amplitudes (voir le [manuscrit](https://arxiv.org/pdf/2412.19150)). Ainsi, pour les grands problèmes, l'Optimized Real Amplitudes nécessite davantage d'exécutions de circuits. Il est susceptible d'être utile pour des problèmes jusqu'à 100 qubits, mais il est recommandé d'être attentif lors du réglage des paramètres ``population_size``. À titre d'exemple de cette mise à l'échelle de ``population_size``, le tableau précédent montre que pour un problème à 84 qubits, l'Optimized Real Amplitudes requiert une ``population_size`` de 120, tandis que pour un problème à 56 qubits, une ``population_size`` de 40 suffit.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

Il est également possible de choisir un ansatz spécifique. L'exemple suivant utilise l'ansatz ``'Tailored'``.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. Exécuter le problème.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. Récupérer les résultats.
Comme mentionné dans la section [Sortie](#output), la fonction retourne un dictionnaire avec les trajectoires d'investissement triées du plus bas au plus élevé selon leur valeur de fonction objectif. Cet ensemble de résultats permet d'identifier la trajectoire au coût le plus bas et ses évaluations d'investissement correspondantes. De plus, il permet d'analyser différentes trajectoires, facilitant la sélection de celles qui correspondent le mieux à des besoins ou objectifs spécifiques. Cette flexibilité garantit que les choix peuvent être adaptés à une grande variété de préférences ou de scénarios.
Commence par présenter la stratégie résultante ayant atteint le coût objectif le plus bas trouvé au cours du processus.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>